# Tiny LLM Decoder 4 (Pytorch End to End - Using Class)

In the fourth version we are implementing Pytorch end to end using class so that we properly scope the variable.

In [12]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [13]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [14]:
# Set Parameters

# random seed
random_seed = 42
torch.manual_seed(random_seed)

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5



## Text Processing and Tokenization

In [15]:
# Tokenization

# Split corpus into words
word_tokens = corpus.split()

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [16]:
vocab_size

19

In [17]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


## Class and Function Setup

In [18]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  next_start_position = start_position + block_size
  return X, Y, next_start_position

In [19]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [20]:
class TinyGPT(nn.Module):

    def __init__(self, vocab_size, d_model):
        super().__init__()

        # Initialize trainable components here
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.WQ = nn.Parameter(torch.randn(d_model, d_model))
        self.WK = nn.Parameter(torch.randn(d_model, d_model))
        self.WV = nn.Parameter(torch.randn(d_model, d_model))

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model, bias=False),
            nn.ReLU(),
            nn.Linear(d_model, d_model, bias=False)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, token_ids):

        # Perform Transformer calculations here
        # Word embeddings + positional encoding
        X_input = self.embeddings(token_ids)
        position = torch.arange(
            len(token_ids),
            dtype=X_input.dtype,
            device=X_input.device
        ).reshape(-1, 1)
        X = X_input + position

        # Q, K, V projections
        Q = X @ self.WQ
        K = X @ self.WK
        V = X @ self.WV

        # Scaled dot-product attention
        attention_score = Q @ K.T
        stable_factor = self.WQ.shape[-1] ** 0.5
        attention_stable = attention_score / stable_factor

        # Causal masking + softmax
        causal_mask = torch.triu(
            torch.full_like(attention_stable, float("-inf")),
            diagonal=1
        )
        attention_stable_mask = attention_stable + causal_mask
        attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

        # Attention output + residual + norm1
        attention_output = attention_softmax @ V

        # Residual Layer
        attention_output_residual = attention_output + X

        # norm1 layer
        attention_output_residual_scaled = self.norm1(attention_output_residual)

        # FFN + residual + norm2
        ffn_1 = self.ffn(attention_output_residual_scaled)

        # Residual 2 Layer
        seond_residual = ffn_1 + attention_output_residual_scaled

        # Norm 2 Layer
        transformer_output = self.norm2(seond_residual)

        # Vocabulary projection
        logits = transformer_output @ self.embeddings.weight.T

        return logits

In [21]:
def llm_train(X_tensor, Y_tensor, model, criterion, optimizer, iters=10):

  for i in range(iters):
    optimizer.zero_grad()

    logits = model(X_tensor)
    loss = criterion(logits, Y_tensor)
    print(f"iters {i} : Loss = {loss.item():.4f}")

    loss.backward()
    optimizer.step()

### Code to Test the number of parameters

In [23]:
model = TinyGPT(vocab_size, d_model)

print(sum(p.numel() for p in model.parameters()))
print(all(p.requires_grad for p in model.parameters()))

172
True


In [24]:
for name, param in model.named_parameters():
    print(name, param.shape)

WQ torch.Size([4, 4])
WK torch.Size([4, 4])
WV torch.Size([4, 4])
embeddings.weight torch.Size([19, 4])
ffn.0.weight torch.Size([4, 4])
ffn.2.weight torch.Size([4, 4])
norm1.weight torch.Size([4])
norm1.bias torch.Size([4])
norm2.weight torch.Size([4])
norm2.bias torch.Size([4])


### Test Class with Input


First Pass

In [25]:
N = get_N(corpus_in_id, block_size=block_size)
N

6

In [26]:
X_id, Y_id, next_start = get_sequence_block(block_size=block_size, start_position=0, corpus_in_id=corpus_in_id)
X_id, Y_id, next_start

([10, 6, 0, 9, 8], [6, 0, 9, 8, 7], 5)

In [27]:
X_tensor = torch.tensor(X_id, dtype=torch.long)
Y_tensor = torch.tensor(Y_id, dtype=torch.long)

In [28]:
model = TinyGPT(vocab_size=vocab_size, d_model=d_model)

logits = model(X_tensor)

print(logits.shape)
print(sum(p.numel() for p in model.parameters()))

torch.Size([5, 19])
172


### Test Model

In [29]:
model = TinyGPT(vocab_size=vocab_size, d_model=d_model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [30]:
llm_train(X_tensor, Y_tensor, model, criterion, optimizer, iters=10)

iters 0 : Loss = 3.2372
iters 1 : Loss = 3.1753
iters 2 : Loss = 3.1193
iters 3 : Loss = 3.0685
iters 4 : Loss = 3.0221
iters 5 : Loss = 2.9798
iters 6 : Loss = 2.9409
iters 7 : Loss = 2.9050
iters 8 : Loss = 2.8717
iters 9 : Loss = 2.8409


In [31]:
llm_train(X_tensor, Y_tensor, model, criterion, optimizer, iters=1000)

iters 0 : Loss = 2.8120
iters 1 : Loss = 2.7849
iters 2 : Loss = 2.7595
iters 3 : Loss = 2.7354
iters 4 : Loss = 2.7127
iters 5 : Loss = 2.6911
iters 6 : Loss = 2.6706
iters 7 : Loss = 2.6510
iters 8 : Loss = 2.6324
iters 9 : Loss = 2.6146
iters 10 : Loss = 2.5975
iters 11 : Loss = 2.5811
iters 12 : Loss = 2.5653
iters 13 : Loss = 2.5502
iters 14 : Loss = 2.5355
iters 15 : Loss = 2.5213
iters 16 : Loss = 2.5076
iters 17 : Loss = 2.4944
iters 18 : Loss = 2.4816
iters 19 : Loss = 2.4691
iters 20 : Loss = 2.4570
iters 21 : Loss = 2.4453
iters 22 : Loss = 2.4339
iters 23 : Loss = 2.4228
iters 24 : Loss = 2.4120
iters 25 : Loss = 2.4015
iters 26 : Loss = 2.3912
iters 27 : Loss = 2.3812
iters 28 : Loss = 2.3714
iters 29 : Loss = 2.3619
iters 30 : Loss = 2.3525
iters 31 : Loss = 2.3433
iters 32 : Loss = 2.3344
iters 33 : Loss = 2.3257
iters 34 : Loss = 2.3171
iters 35 : Loss = 2.3086
iters 36 : Loss = 2.3004
iters 37 : Loss = 2.2923
iters 38 : Loss = 2.2843
iters 39 : Loss = 2.2765
iters 40 :

### Train the Entire Corpus

In [32]:
N = get_N(corpus_in_id, block_size=block_size)
N

6

In [33]:
model = TinyGPT(vocab_size=vocab_size, d_model=d_model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [35]:
for i in range(N):
  if i == 0:
    next_start = 0

  X_id, Y_id, next_start = get_sequence_block(block_size=block_size, start_position=next_start, corpus_in_id=corpus_in_id)
  X_tensor = torch.tensor(X_id, dtype=torch.long)
  Y_tensor = torch.tensor(Y_id, dtype=torch.long)
  print(f"Training block {i}")
  llm_train(X_tensor, Y_tensor, model, criterion, optimizer, iters=10)


Training block 0
iters 0 : Loss = 2.8728
iters 1 : Loss = 2.8384
iters 2 : Loss = 2.8061
iters 3 : Loss = 2.7758
iters 4 : Loss = 2.7473
iters 5 : Loss = 2.7207
iters 6 : Loss = 2.6956
iters 7 : Loss = 2.6720
iters 8 : Loss = 2.6498
iters 9 : Loss = 2.6290
Training block 1
iters 0 : Loss = 3.0525
iters 1 : Loss = 3.0272
iters 2 : Loss = 3.0025
iters 3 : Loss = 2.9786
iters 4 : Loss = 2.9552
iters 5 : Loss = 2.9325
iters 6 : Loss = 2.9103
iters 7 : Loss = 2.8886
iters 8 : Loss = 2.8675
iters 9 : Loss = 2.8469
Training block 2
iters 0 : Loss = 3.1968
iters 1 : Loss = 3.1644
iters 2 : Loss = 3.1362
iters 3 : Loss = 3.1115
iters 4 : Loss = 3.0898
iters 5 : Loss = 3.0704
iters 6 : Loss = 3.0528
iters 7 : Loss = 3.0366
iters 8 : Loss = 3.0215
iters 9 : Loss = 3.0073
Training block 3
iters 0 : Loss = 3.4783
iters 1 : Loss = 3.4548
iters 2 : Loss = 3.4320
iters 3 : Loss = 3.4099
iters 4 : Loss = 3.3884
iters 5 : Loss = 3.3677
iters 6 : Loss = 3.3476
iters 7 : Loss = 3.3280
iters 8 : Loss = 3.3

In [36]:
model.eval()

with torch.no_grad():
    for i in range(N):
        start = i * block_size

        X_id, Y_id, _ = get_sequence_block(
            block_size=block_size,
            start_position=start,
            corpus_in_id=corpus_in_id
        )

        X = torch.tensor(X_id, dtype=torch.long)
        Y = torch.tensor(Y_id, dtype=torch.long)

        logits = model(X)
        loss = criterion(logits, Y)

        print(f"Block {i}: Loss = {loss.item():.4f}")

Block 0: Loss = 2.6754
Block 1: Loss = 2.6578
Block 2: Loss = 3.0945
Block 3: Loss = 3.1751
Block 4: Loss = 2.6288
Block 5: Loss = 2.9695


This is the wrong approach, as we train each block 10 times before moving to next block. This results in block 5 performs better than block 0. The better approach is do 1 training pass for each block before moving to epoch 2. We add epoch on the outer loop.

## END